# Training and evaluating a vanilla neural network

In [19]:
import json
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import numpy as np

data_path = "../data"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

VANILLA_NN_MODEL_PATH = "../models/vanilla_nn/3H1L_vanilla_nn_model.pth"
SENTENCE_EMBED_TRAIN_VECTOR_PATH = "../models/vanilla_nn/3H1L_train_nn_sentence_embeddings.pkl"
SENTENCE_EMBED_VALIDATE_VECTOR_PATH = "../models/vanilla_nn/3H1L_validate_nn_sentence_embeddings.pkl"
SENTENCE_EMBED_TEST_VECTOR_PATH = "../models/vanilla_nn/3H1L_test_nn_sentence_embeddings.pkl"


### Transform data into appropriate format

Create Python list of dicts

In [2]:
def flatten_transform(target_set):

    modified_set = []

    for item in target_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_nn_ready = flatten_transform(train_nn_set)
validate_set_nn_ready = flatten_transform(validate_set)
test_set_nn_ready = flatten_transform(test_set)

Transform into dataframe and create X and y classes

In [3]:
train_nn_df = pd.DataFrame(train_set_nn_ready)
X_train_nn_text = train_nn_df["Headline"]
y_train = train_nn_df["Fake"]

validate_nn_df = pd.DataFrame(validate_set_nn_ready)
X_validate_nn_text = validate_nn_df["Headline"]
y_validate = validate_nn_df["Fake"]

test_df = pd.DataFrame(test_set_nn_ready)
X_test_nn_text = test_df["Headline"]

### Vanilla neural network

Get sentence embeddings

In [9]:
try:
    X_train_nn_SE = joblib.load(SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    X_validate_nn_SE = joblib.load(SENTENCE_EMBED_VALIDATE_VECTOR_PATH)
    X_test_nn_SE = joblib.load(SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Loaded model and embeddings from disk.")

except FileNotFoundError:

    print("No saved embeddings found, creating new ones...")

    # Create embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Embed both train and test sets
    X_train_nn_SE = model.encode(list(X_train_nn_text))
    X_validate_nn_SE = model.encode(list(X_validate_nn_text))
    X_test_nn_SE = model.encode(list(X_test_nn_text))

    joblib.dump(X_train_nn_SE, SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    joblib.dump(X_validate_nn_SE, SENTENCE_EMBED_VALIDATE_VECTOR_PATH)
    joblib.dump(X_test_nn_SE, SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Model and embeddings saved.")

print("Embeddings matrix shape (train):", X_train_nn_SE.shape)
print("Embeddings matrix shape (validate):", X_validate_nn_SE.shape)
print("Embeddings matrix shape (test):", X_test_nn_SE.shape)

Loaded model and embeddings from disk.
Embeddings matrix shape (train): (812, 384)
Embeddings matrix shape (validate): (268, 384)
Embeddings matrix shape (test): (268, 384)


Convert to PyTorch tensors and package into batches

In [10]:
X_train_tensor = torch.FloatTensor(X_train_nn_SE)
y_train_tensor = torch.FloatTensor(y_train.to_numpy()).unsqueeze(1)
X_validate_tensor = torch.FloatTensor(X_validate_nn_SE)
y_validate_tensor = torch.FloatTensor(y_validate.to_numpy()).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test_nn_SE)
# y_test_tensor = torch.FloatTensor(y_test.to_numpy()).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

C:\Users\popro\AppData\Local\Temp\ipykernel_1760\3640200872.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train_tensor = torch.FloatTensor(y_train.to_numpy()).unsqueeze(1)


Define neural network

In [11]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.layer3(output2)
        return output3

Instantiate the model and define loss function

In [12]:
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

loss_function = nn.BCEWithLogitsLoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

Train model

In [39]:
MAX_EPOCHS = 20

best_val_loss = float("inf")
counter = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    losses = []
    for batch_X, batch_y in train_loader:
        optimiser.zero_grad()
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        losses.append(loss.item())
        loss.backward()
        optimiser.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_validate_tensor)
        val_loss = loss_function(val_outputs, y_validate_tensor)

    print(f'Epoch [{epoch+1}/{MAX_EPOCHS}], Train Loss: {(sum(losses) / len(losses)):.3f}, Val Loss: {val_loss.item():.3f}')

    current_val_loss = val_loss.item()

    if current_val_loss < best_val_loss:
        print("New best validation loss found, saving model")
        best_val_loss = current_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1
        if counter == 5:
            print("Early stopping")
            break

model.load_state_dict(best_model_state)
    


Epoch [1/20], Train Loss: 0.640, Val Loss: 0.593
New best validation loss found, saving model
Epoch [2/20], Train Loss: 0.557, Val Loss: 0.542
New best validation loss found, saving model
Epoch [3/20], Train Loss: 0.506, Val Loss: 0.528
New best validation loss found, saving model
Epoch [4/20], Train Loss: 0.474, Val Loss: 0.523
New best validation loss found, saving model
Epoch [5/20], Train Loss: 0.437, Val Loss: 0.534
Epoch [6/20], Train Loss: 0.396, Val Loss: 0.540
Epoch [7/20], Train Loss: 0.354, Val Loss: 0.567
Epoch [8/20], Train Loss: 0.294, Val Loss: 0.613
Epoch [9/20], Train Loss: 0.237, Val Loss: 0.634
Early stopping


<All keys matched successfully>

Save model

In [40]:
torch.save(model.state_dict(), VANILLA_NN_MODEL_PATH)

Load model

In [13]:
model = NeuralNetwork(input_size)
model.load_state_dict(torch.load(VANILLA_NN_MODEL_PATH))
model.eval()

NeuralNetwork(
  (layer1): Linear(in_features=384, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (layer3): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
)

Evaluate

In [14]:
with torch.no_grad():
    y_fake_logits = model(X_test_tensor)
# y_fake_logits

In [15]:
y_fake_probs = torch.sigmoid(y_fake_logits)
# y_fake_probs

In [16]:
test_df["Fake Logit VNN"] = y_fake_logits
test_by_quiz_df = test_df.groupby("Date")

correct = 0

for date, quiz_df in test_by_quiz_df:
    predicted_index = quiz_df["Fake Logit VNN"].idxmax()
    actual_index = quiz_df["Fake"].idxmax()

    if predicted_index == actual_index:
        correct += 1

accuracy_vanilla_nn = correct / len(test_set)

print(f"Quiz-level accuracy: {accuracy_vanilla_nn:.2%}")

Quiz-level accuracy: 46.27%


In [20]:
# Reshape headline-level probabilities into 67 quizzes × 4 headlines
fake_probs_MLP = y_fake_probs.cpu().numpy().reshape(-1, 4)

# Predicted fake headline position (0–3)
y_pred_quiz_MLP = np.argmax(fake_probs_MLP, axis=1)

# Confidence = fake probability assigned to the selected headline
confidence_MLP = np.max(fake_probs_MLP, axis=1)

# Dates from the fixed test set
test_dates = [quiz["date"] for quiz in test_set]

# Create prediction dataframe
predictions_MLP = pd.DataFrame({
    "date": test_dates,
    "prediction": y_pred_quiz_MLP,
    "confidence": confidence_MLP
})

predictions_MLP.to_csv("../data/predictions_minilm_mlp.csv", index=False)

In [17]:
test_df

,Date,Headline,Fake,Fake Logit VNN
0,2019-11-23,Taylor Swift fans boycott J.Crew over private ...,1,-0.380976
1,2019-11-23,Cambridge University students cry fowl over 17...,0,-1.024492
2,2019-11-23,Why the world is running out of sand,0,-0.867993
3,2019-11-23,Manager drives from New York to Pennsylvania t...,0,-1.166699
4,2020-01-18,Yamaha warns musicians not to climb in instrum...,0,-1.274740
...,...,...,...,...
263,2026-07-23,Ed Harris slams his _Dutton Ranch_ role: “Get ...,0,-1.092670
264,2026-08-27,The new cure for career burnout: clown school,0,-0.291697
265,2026-08-27,A simple bean is changing beauty pageants,1,-0.620151
266,2026-08-27,Denmark’s PM blasts plastic surgeon for his un...,0,-2.480407


In [79]:
predictions = {}

for date, quiz_df in test_by_quiz_df:
    predictions[date] = quiz_df["Fake Logit VNN"].idxmax()

joblib.dump(predictions, f"{data_path}/vnn_predictions.pkl")

['../data/vnn_predictions.pkl']

In [80]:
nn_predictions = joblib.load(f"{data_path}/vnn_predictions.pkl")
se_predictions = joblib.load(f"{data_path}/se_lr_predictions.pkl")

same_prediction = sum(
    nn_predictions[date] == se_predictions[date]
    for date in nn_predictions
)

print(f"Same prediction: {same_prediction}/{len(nn_predictions)}")

Same prediction: 51/67


In [ ]:
lr_df = joblib.load(f"{data_path}/lr_df.pkl")
lr_df["Fake Prob VNN"] = torch.sigmoid(y_fake_logits)
comparison_df = lr_df

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

comparison_df


In [22]:
joblib.dump(comparison_df, f"{data_path}/comparison_df.pkl")

['../data/comparison_df.pkl']